# Microsoft Agent Framework를 활용한 휴먼 인 더 루프 워크플로우

## 🎯 학습 목표

이 노트북에서는 Microsoft Agent Framework의 `RequestInfoExecutor`를 사용하여 **휴먼 인 더 루프(human-in-the-loop)** 워크플로우를 구현하는 방법을 배웁니다. 이 강력한 패턴은 AI 워크플로우를 일시 중지하여 인간의 입력을 수집할 수 있게 하며, 에이전트를 상호작용 가능하게 만들고 중요한 결정을 사람의 통제 하에 둘 수 있습니다.

## 🔄 휴먼 인 더 루프란 무엇인가?

<strong>휴먼 인 더 루프(HITL)</strong>는 AI 에이전트가 실행을 일시 중지하고 계속하기 전에 인간의 입력을 요청하는 설계 패턴입니다. 이는 다음과 같은 경우에 필수적입니다:

- ✅ **중요한 결정** - 중요한 행동 전에 인간의 승인 받기
- ✅ **애매한 상황** - AI가 불확실할 때 인간이 명확히 하도록 하기
- ✅ **사용자 선호도** - 사용자에게 여러 옵션 중 선택 요청하기
- ✅ **규정 준수 및 안전** - 규제 작업에 대해 인간의 감독 보장하기
- ✅ **인터랙티브 경험** - 사용자 입력에 반응하는 대화형 에이전트 구축하기

## 🏗️ Microsoft Agent Framework에서 작동 원리

이 프레임워크는 HITL을 위해 세 가지 주요 구성 요소를 제공합니다:

1. **`RequestInfoExecutor`** - 워크플로우를 일시 중지하고 `RequestInfoEvent`를 내보내는 특수 실행기
2. **`RequestInfoMessage`** - 인간에게 보내는 타입화된 요청 페이로드의 기본 클래스
3. **`RequestResponse`** - `request_id`를 사용해 인간 응답과 원래 요청을 연동

**워크플로우 패턴:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 예제: 사용자 확인이 포함된 호텔 예약

조건부 워크플로우를 기반으로 대체 목적지 제안 <strong>전</strong>에 인간 확인을 추가해 보겠습니다:

1. 사용자가 목적지(예: "파리")를 요청함
2. `availability_agent`가 객실 이용 가능 여부 확인
3. **객실이 없으면** → `confirmation_agent`가 "대체 옵션을 보시겠습니까?"라고 문의
4. `RequestInfoExecutor`를 사용해 워크플로우 **일시 중지**
5. **사람이 콘솔 입력으로** "예" 또는 "아니오" 응답
6. `decision_manager`가 응답에 따라 경로 지정:
   - <strong>예</strong> → 대체 목적지 표시
   - <strong>아니오</strong> → 예약 요청 취소
7. 최종 결과 표시

이것은 사용자에게 에이전트 제안에 대한 통제권을 주는 방법을 보여줍니다!

---

시작해 봅시다! 🚀


## 1단계: 필요한 라이브러리 가져오기

표준 에이전트 프레임워크 구성 요소와 함께 <strong>인간 참여 루프 특정 클래스</strong>를 가져옵니다:
- `RequestInfoExecutor` - 인간 입력을 위해 워크플로우를 일시 중지하는 실행기
- `RequestInfoEvent` - 인간 입력 요청 시 발생하는 이벤트
- `RequestInfoMessage` - 유형화된 요청 페이로드의 기본 클래스
- `RequestResponse` - 인간 응답을 요청과 연관시킴
- `WorkflowOutputEvent` - 워크플로우 출력을 감지하는 이벤트


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## 2단계: 구조화된 출력을 위한 Pydantic 모델 정의

이 모델들은 에이전트가 반환할 <strong>스키마</strong>를 정의합니다. 조건부 워크플로에서 사용한 모든 모델을 유지하고 다음을 추가합니다:

**Human-in-the-Loop를 위한 신규 모델:**
- `HumanFeedbackRequest` - `RequestInfoMessage`의 서브클래스로, 사람에게 전송되는 요청 페이로드를 정의합니다
  - `prompt`(묻고자 하는 질문)와 `destination`(이용할 수 없는 도시에 대한 컨텍스트)을 포함합니다


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## 3단계: 호텔 예약 도구 만들기

조건부 워크플로에서 사용한 것과 동일한 도구 - 목적지에 객실이 있는지 확인합니다.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## 4단계: 라우팅을 위한 조건 함수 정의

인간 참여 워크플로우를 위해 <strong>네 개의 조건 함수</strong>가 필요합니다:

**조건부 워크플로우에서 가져온 것:**
1. `has_availability_condition` - 호텔이 이용 가능할 때 라우팅
2. `no_availability_condition` - 호텔이 이용 불가능할 때 라우팅

**인간 참여를 위한 새로운 조건:**
3. `user_wants_alternatives_condition` - 사용자가 대안에 "예"라고 말할 때 라우팅
4. `user_declines_alternatives_condition` - 사용자가 대안에 "아니요"라고 말할 때 라우팅


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## 5단계: Decision Manager Executor 생성하기

이것이 <strong>사람 개입형 패턴의 핵심</strong>입니다! `DecisionManager`는 다음을 수행하는 맞춤형 `Executor`입니다:

1. `RequestResponse` 객체를 통해 **사람의 피드백을 받음**
2. 사용자의 결정(예/아니요)을 <strong>처리함</strong>
3. 적절한 에이전트에게 메시지를 전송하여 **워크플로우 경로를 지정함**

주요 특징:
- `@handler` 데코레이터를 사용하여 메서드를 워크플로우 단계로 노출
- 원래 요청과 사용자의 답변을 모두 포함하는 `RequestResponse[HumanFeedbackRequest, str]` 수신
- 조건 함수를 트리거하는 간단한 "yes" 또는 "no" 메시지 전달


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## 6단계: 사용자 정의 디스플레이 실행기 만들기

조건부 워크플로우의 동일한 디스플레이 실행기 - 최종 결과를 워크플로우 출력으로 내보냅니다.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## 7단계: 환경 변수 로드

LLM 클라이언트(Microsoft Foundry / Azure OpenAI)를 구성합니다.


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## 8단계: AI 에이전트 및 실행기 생성

우리는 <strong>여섯 개의 워크플로우 구성 요소</strong>를 만듭니다:

**에이전트(AgentExecutor로 래핑됨):**
1. **availability_agent** - 도구를 사용하여 호텔 이용 가능 여부 확인
2. **confirmation_agent** - 🆕 인간 확인 요청 준비
3. **alternative_agent** - 대체 도시 제안 (사용자가 예라고 말할 때)
4. **booking_agent** - 예약 권장 (객실 이용 가능 시)
5. **cancellation_agent** - 🆕 취소 메시지 처리 (사용자가 아니라고 할 때)

**특별 실행기:**
6. **request_info_executor** - 🆕 인간 입력을 위해 워크플로우를 일시 중지하는 `RequestInfoExecutor`
7. **decision_manager** - 🆕 인간 응답을 기반으로 라우팅하는 맞춤 실행기 (위에서 이미 정의됨)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## 9단계: 인간 참여 루프와 함께 워크플로우 구축

이제 인간 참여 루프 경로를 포함한 <strong>조건부 라우팅</strong>으로 워크플로우 그래프를 구성합니다:

**워크플로우 구조:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**핵심 엣지:**
- `availability_agent → confirmation_agent` (방이 없을 때)
- `confirmation_agent → prepare_human_request` (타입 변환)
- `prepare_human_request → request_info_executor` (사람 대기)
- `request_info_executor → decision_manager` (항상 - RequestResponse 제공)
- `decision_manager → alternative_agent` (사용자가 "예"라고 할 때)
- `decision_manager → cancellation_agent` (사용자가 "아니오"라고 할 때)
- `availability_agent → booking_agent` (방이 있을 때)
- 모든 경로는 `display_result`에서 종료됨


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## 10단계: 테스트 케이스 1 실행 - 가용성이 없는 도시 (사람 확인이 필요한 파리)

이 테스트는 <strong>완전한 인간 개입 주기</strong>를 보여줍니다:

1. 파리의 호텔 요청
2. availability_agent 확인 → 방 없음
3. confirmation_agent가 사람 대상 질문 생성
4. request_info_executor가 <strong>워크플로우를 일시 중지</strong>하고 `RequestInfoEvent` 발생
5. **애플리케이션이 이벤트를 감지하고 콘솔에서 사용자에게 알림**
6. 사용자가 "yes" 또는 "no" 입력
7. 애플리케이션이 `send_responses_streaming()`으로 응답 전송
8. decision_manager가 응답에 따라 경로 지정
9. 최종 결과 표시

**핵심 패턴:**
- 첫 번째 반복은 `workflow.run_stream()` 사용
- 이후 반복은 `workflow.send_responses_streaming(pending_responses)` 사용
- 인간 입력 필요 시 `RequestInfoEvent` 수신 대기
- 최종 결과 캡처를 위해 `WorkflowOutputEvent` 수신 대기


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## 11단계: 테스트 케이스 2 실행 - 방 있음 도시 (스톡홀름 - 사용자 입력 불필요)

이 테스트는 방이 있는 경우의 <strong>직접 경로</strong>를 보여줍니다:

1. 스톡홀름 호텔 요청
2. availability_agent 확인 → 방 있음 ✅
3. booking_agent가 예약 제안
4. display_result가 확인 메시지 표시
5. **사용자 입력 필요 없음!**

방이 있을 때 워크플로가 인간 참여 경로를 완전히 우회합니다.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## 주요 내용 및 휴먼 인 더 루프 모범 사례

### ✅ 배운 내용:

#### 1. **RequestInfoExecutor 패턴**
Microsoft Agent Framework의 휴먼 인 더 루프 패턴은 세 가지 핵심 구성 요소를 사용합니다:
- `RequestInfoExecutor` - 워크플로우를 일시 중지하고 이벤트를 발생시킴
- `RequestInfoMessage` - 타입이 지정된 요청 페이로드의 기본 클래스 (이 클래스를 상속하세요!)
- `RequestResponse` - 인간의 응답을 원래 요청과 연관시킴

**중요한 이해:**
- `RequestInfoExecutor`는 스스로 입력을 수집하지 않음 - 단지 워크플로우를 일시 중지합니다
- 애플리케이션 코드는 `RequestInfoEvent`를 수신하고 입력을 수집해야 합니다
- `send_responses_streaming()`을 호출하여 `request_id`를 사용자 답변에 매핑한 딕셔너리를 전달해야 합니다

#### 2. **스트리밍 실행 패턴**
```python
# 첫 번째 반복
stream = workflow.run_stream(initial_request)

# 이후 반복(사용자 입력 후)
stream = workflow.send_responses_streaming(pending_responses)

# 항상 이벤트 처리
events = [event async for event in stream]
```

#### 3. **이벤트 기반 아키텍처**
다음 특정 이벤트를 수신하여 워크플로우를 제어하세요:
- `RequestInfoEvent` - 인간 입력이 필요함 (워크플로우 일시 중지)
- `WorkflowOutputEvent` - 최종 결과 사용 가능 (워크플로우 완료)
- `WorkflowStatusEvent` - 상태 변경 (진행 중, 대기 중 요청 있음 등)

#### 4. **@handler가 있는 커스텀 실행기**
`DecisionManager`는 다음과 같이 실행기를 만드는 방법을 보여줍니다:
- `@handler` 데코레이터를 사용해 메서드를 워크플로우 단계로 노출
- 타입이 지정된 메시지 수신 (예: `RequestResponse[HumanFeedbackRequest, str]`)
- 메시지를 다른 실행기로 보내 워크플로우 라우팅
- `WorkflowContext`를 통해 컨텍스트 접근

#### 5. **인간 결정에 따른 조건부 라우팅**
인간 응답을 평가하는 조건 함수를 만들 수 있습니다:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 실제 적용 사례:

1. **승인 워크플로우**
   - 비용 보고서 처리 전에 관리자 승인 받기
   - 자동 이메일 발송 전 인간 검토 필요
   - 고액 거래 실행 전 확인

2. **콘텐츠 검토**
   - 의심되는 콘텐츠를 인간 검토 대상으로 지정
   - 엣지 케이스에 대해 검토자에게 최종 결정 요청
   - AI 신뢰도가 낮을 때 인간에게 에스컬레이션

3. **고객 서비스**
   - 일상적인 질문은 AI가 자동 처리
   - 복잡한 문제는 인간 상담원에게 에스컬레이션
   - 고객에게 인간 상담원 연결 여부 문의

4. **데이터 처리**
   - 애매한 데이터 항목은 인간에게 해결 요청
   - 불분명한 문서 해석을 AI가 했을 때 인간 확인
   - 여러 가능한 해석 중 사용자 선택 허용

5. **안전-critical 시스템**
   - 되돌릴 수 없는 작업 전 인간 확인 요구
   - 민감한 데이터 접근 전 승인 요구
   - 규제 산업(의료, 금융)에서 결정 확인

6. **인터랙티브 에이전트**
   - 후속 질문을 하는 대화형 봇 구축
   - 복잡한 프로세스를 안내하는 마법사 제작
   - 단계별로 인간과 협력하는 에이전트 설계

### 🔄 비교: 인간 참여 유무

| 기능 | 조건부 워크플로우 | 휴먼 인 더 루프 워크플로우 |
|---------|---------------------|---------------------------|
| <strong>실행</strong> | 단일 `workflow.run()` | `run_stream()` + `send_responses_streaming()` 루프 |
| **사용자 입력** | 없음 (완전 자동) | `input()` 혹은 UI를 통한 상호작용 프롬프트 |
| **구성 요소** | 에이전트 + 실행기 | + RequestInfoExecutor + DecisionManager |
| <strong>이벤트</strong> | AgentExecutorResponse만 | RequestInfoEvent, WorkflowOutputEvent 등 |
| **일시 중지** | 일시 중지 없음 | RequestInfoExecutor에서 워크플로우 일시 중지 |
| **인간 제어** | 인간 제어 없음 | 인간이 주요 결정을 내림 |
| **사용 사례** | 자동화 | 협력 및 감독 |

### 🚀 고급 패턴:

#### 다중 인간 결정 지점
동일 워크플로우에서 여러 `RequestInfoExecutor` 노드를 가질 수 있습니다:
```python
.add_edge(agent1, request_info_1)  # 첫 번째 인간 결정
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # 두 번째 인간 결정
.add_edge(decision_manager_2, final_agent)
```

#### 타임아웃 처리
인간 응답에 대한 타임아웃 구현:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # 기본값은 안전한 옵션으로 설정
```

#### 풍부한 UI 통합
`input()` 대신 웹 UI, Slack, Teams 등과 통합:
```python
if isinstance(event, RequestInfoEvent):
    # 사용자가 선호하는 채널로 알림을 전송하세요
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### 조건부 휴먼 인 더 루프
특정 상황에서만 인간 입력 요청:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # 신뢰도가 낮거나 값이 높을 경우에만 사람에게 경로를 지정합니다
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ 모범 사례:

1. **항상 RequestInfoMessage를 서브클래스화 하세요**
   - 타입 안전성과 검증 제공
   - UI 렌더링을 위한 풍부한 컨텍스트 가능
   - 각 요청 유형의 의도 명확화

2. **설명적인 프롬프트 사용**
   - 요청하는 내용의 컨텍스트 포함
   - 각 선택의 결과 설명
   - 질문을 간단하고 명확하게 유지

3. **예상치 못한 입력 처리**
   - 사용자 응답 검증
   - 유효하지 않은 입력에 대한 기본값 제공
   - 명확한 오류 메시지 제공

4. **요청 ID 추적**
   - request_id와 응답의 연관성 활용
   - 상태를 수동으로 관리하지 마세요

5. **논블로킹 설계**
   - 입력 대기 중 스레드를 차단하지 마세요
   - 전반적으로 비동기 패턴 사용
   - 동시 워크플로우 인스턴스 지원

### 📚 관련 개념:

- **에이전트 미들웨어** - 에이전트 호출 가로채기 (이전 노트북)
- **워크플로우 상태 관리** - 실행 간 워크플로우 상태 유지
- **멀티 에이전트 협업** - 휴먼 인 더 루프와 에이전트 팀 결합
- **이벤트 기반 아키텍처** - 이벤트로 반응형 시스템 구축

---

### 🎓 축하합니다!

Microsoft Agent Framework로 휴먼 인 더 루프 워크플로우를 마스터했습니다! 이제 다음을 할 수 있습니다:
- ✅ 인간 입력을 수집하기 위해 워크플로우를 일시 중지
- ✅ RequestInfoExecutor와 RequestInfoMessage 사용
- ✅ 이벤트와 함께 스트리밍 실행 처리
- ✅ @handler를 사용해 커스텀 실행기 생성
- ✅ 인간 결정에 따라 워크플로우 라우팅
- ✅ 인간과 협력하는 인터랙티브 AI 에이전트 구축

**이것은 신뢰할 수 있고 제어 가능한 AI 시스템을 구축하는 데 필수적인 패턴입니다!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
